In [ ]:
# ---------------------------------------------------------------------
# merge_results.py
# ---------------------------------------------------------------------
# Combine all chunk-level simulation outputs into one full dataset.
# Adds (x, y, z) detector coordinates from CSV input files.
# ---------------------------------------------------------------------

import os
import pandas as pd
from pathlib import Path

# --- Paths ---
BASE_DIR = Path.cwd().resolve().parents[0]
DATA_DIR = BASE_DIR / "data"
FILE = "sbi_n300000_low_shm.csv"  # ------------------------------------------------------------------------------------------
DATASET_NAME = Path(FILE).stem
S1S2_DIR = DATA_DIR / DATASET_NAME / "s1s2"

# Path to coordinate input CSVs (change as needed)
COORD_INPUT_DIR = Path("/scratch/midway3/nreus/dark_matter_sbi/sbi_n300000_low_shm/csv_input")  # ----------------------------

os.makedirs(S1S2_DIR, exist_ok=True)


def merge_all_chunks():
    # Find all chunk output files
    s1s2_files = sorted(S1S2_DIR.glob("s1s2_chunk_*.csv"))
    print(f"[merge_results] Found {len(s1s2_files)} chunk outputs in {S1S2_DIR}")

    dfs = []
    for s1s2_file in s1s2_files:
        chunk_id = s1s2_file.stem.split("_")[-1]
        coord_file = COORD_INPUT_DIR / f"chunk_{chunk_id}.csv"

        if not coord_file.exists():
            print(f"[warning] No coordinate file found for chunk {chunk_id}, skipping coords.")
            s1s2_df = pd.read_csv(s1s2_file)
            dfs.append(s1s2_df)
            continue

        # Read both dataframes
        s1s2_df = pd.read_csv(s1s2_file)
        coord_df = pd.read_csv(coord_file)

        # Expected coordinate columns (adjust if different)
        coord_cols = ["xp", "yp", "zp"]
        if not coord_cols:
            print(f"[warning] No coordinate columns found in {coord_file}, skipping coords.")
            dfs.append(s1s2_df)
            continue

        # Sanity check
        if len(coord_df) != len(s1s2_df):
            print(f"[warning] Length mismatch in chunk {chunk_id}: "
                  f"{len(coord_df)} coords vs {len(s1s2_df)} events. Skipping coords.")
            dfs.append(s1s2_df)
            continue

        # Merge by index
        merged_df = pd.concat([s1s2_df.reset_index(drop=True),
                               coord_df[coord_cols].reset_index(drop=True)], axis=1)

        dfs.append(merged_df)

    # Combine all chunks
    full_df = pd.concat(dfs, ignore_index=True)

    # Save final dataset
    out_path = DATA_DIR / DATASET_NAME / "s1s2_n300000_low_shm_full.csv" # --------------------------------------------------------------------------
    full_df.to_csv(out_path, index=False)
    print(f"[merge_results] Saved full dataset with {len(full_df)} entries → {out_path}")


if __name__ == "__main__":
    merge_all_chunks()